<a href="https://colab.research.google.com/github/netsetos/agentic-ai-weekend-gcp-learners/blob/main/module-12-production-deploy/lesson-12.1-infra-setup/notebooks/GCP_Capstone_12.1_InfraSetup.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 12.1 Infrastructure & Project Setup — The Plan Is Empty
**Netsetos GenAI Engineering — GCP Capstone** · Module 12 · rebuilt on the live lane, 10 September 2026

Terraform the foundation - and then prove the foundation is what Terraform says. This notebook is where the kit's `terraform/*.tf` come from (the heredocs at the end are what the extractor reads), and the lane on `documind-ai-YOUR-ID` was stood up from exactly them by `make up`. The story now runs against that lane: the state read from the tfstate bucket, the project read back through IAM, Storage, Secret Manager and Billing until the two agree, the two profiles counted from the clone, and the cost controls this lesson provisions since Module 11's evening. The proof that the code and the project agree is an empty plan; `make drift` is where it is run.


## Setup


In [ ]:
!pip install -q google-genai==2.22.0 google-cloud-storage==3.13.1 requests==2.34.2

from google.colab import auth
auth.authenticate_user()

PROJECT_ID = "documind-ai-YOUR-ID"   # CHANGE THIS: the project the lane runs in (make up, lesson 4.8)
REGION     = "us-central1"
TENANT     = "acme"
KIT        = "/content/agentic-ai-weekend-gcp-learners"   # the kit: deploy/shared is the tool layer every lesson on the lane imports
BRANCH     = "main"        # the learner repo's branch: the notebooks and the kit (deploy/) ship there together

import os, subprocess, sys
import google.auth
from google.auth.transport.requests import AuthorizedSession
from google import genai
from google.genai import types

if not os.path.isdir(KIT):
    subprocess.run(["git", "clone", "--depth", "1", "-q", "-b", BRANCH,
                    "https://github.com/netsetos/agentic-ai-weekend-gcp-learners", KIT], check=True)
sys.path.insert(0, f"{KIT}/deploy")                   # `from shared import ...` - the same layer every service imports

# The lane's URLs are deterministic: service name + project NUMBER (eventarc.tf builds them the same way).
creds, _ = google.auth.default()
NUMBER = AuthorizedSession(creds).get(
    f"https://cloudresourcemanager.googleapis.com/v1/projects/{PROJECT_ID}").json()["projectNumber"]
API_URL       = f"https://documind-api-{NUMBER}.{REGION}.run.app"
UPLOAD_BUCKET = f"{PROJECT_ID}-uploads"     # storage.tf: the bucket eventarc.tf watches - the corpus, media included
MEDIA_BUCKET  = f"{PROJECT_ID}-media"       # storage.tf: generated assets, 30-day lifecycle (a cache, not a record)
DATASETS      = f"{PROJECT_ID}-datasets"    # storage.tf (Module 10): the frozen tuning dataset and 10.5's GGUF
GATEWAY_URL   = f"https://documind-gateway-{NUMBER}.{REGION}.run.app"   # 11.3: LiteLLM on Cloud Run, behind IAM (make deploy-gateway)
SLM_URL       = f"https://documind-slm-{NUMBER}.{REGION}.run.app"       # 11.4: Ollama on an L4, min-instances 0 (make deploy-slm)
os.environ.update({
    "GOOGLE_CLOUD_PROJECT": PROJECT_ID,
    "GOOGLE_CLOUD_LOCATION": "global",              # Gemini 3.x generation is served from the global endpoint
    "GOOGLE_GENAI_USE_VERTEXAI": "TRUE",
    "DOCUMIND_PROFILE": "gcp",
    "RAG_API_URL": API_URL,
    "RAG_TIMEOUT_S": "90",                          # 7.2's finding: a cold API takes longer than the default 20 s
    # A notebook has no metadata server to be anyone with: the kit mints its ID tokens AS this roster
    # member (7.1). On Cloud Run the service's own account is the identity and nothing is set.
    "DOCUMIND_IMPERSONATE_SA": f"documind-ui-sa@{PROJECT_ID}.iam.gserviceaccount.com",
})
MEMBER_SA   = os.environ["DOCUMIND_IMPERSONATE_SA"]
OUTSIDER_SA = f"documind-outsider-sa@{PROJECT_ID}.iam.gserviceaccount.com"   # IAM admits it, no roster does (4.8, 7.2)

from shared import documind_tools    # THE one retrieve(). Imported, never pasted - the contract gate fails a paste.
gen = genai.Client(enterprise=True, project=PROJECT_ID, location="global")   # every generate_content in this lesson
TFSTATE = f"{PROJECT_ID}-tfstate"   # the state bucket: created once by hand (gsutil mb), then Terraform owns everything else

print("kit:", KIT, "| API:", API_URL, "| gateway:", GATEWAY_URL, "| slm:", SLM_URL)


## Cell 1: The API, the gateway and the SLM, called the way the lane calls them


In [ ]:
import json, requests, time, subprocess, datetime
from google.cloud import storage

# THE API, THE GATEWAY AND THE SLM, CALLED THE WAY THE LANE CALLS THEM: one ID token per request, minted AS the roster
# member for the service's own URL (the kit mints it: documind_tools._id_token). Every service on the lane is behind
# Cloud Run IAM; there is no master key and no API key to paste.
def api(path: str, body: dict | None = None, base: str | None = None, timeout: int = 120, token: str | None = "member") -> tuple[int, dict | str]:
    """POST one API route (or a candidate revision's, with base=) - as documind-ui-sa by default, with token=None as
    nobody, or with a token minted as another account. Returns (status, json-or-text)."""
    url = (base or API_URL).rstrip("/")
    headers = {}
    if token == "member":
        headers["Authorization"] = f"Bearer {documind_tools._id_token(API_URL)}"     # the audience is the canonical URL
    elif token:
        headers["Authorization"] = f"Bearer {token}"
    r = requests.post(f"{url}{path}", json=body, headers=headers, timeout=timeout)
    try:
        return r.status_code, r.json()
    except ValueError:
        return r.status_code, r.text[:400]

def api_get(path: str, base: str | None = None, timeout: int = 60) -> tuple[int, dict | str]:
    """GET one API route as the roster member: /version, /health."""
    url = (base or API_URL).rstrip("/")
    r = requests.get(f"{url}{path}", headers={"Authorization": f"Bearer {documind_tools._id_token(API_URL)}"}, timeout=timeout)
    try:
        return r.status_code, r.json()
    except ValueError:
        return r.status_code, r.text[:400]

def gateway(model: str, content: str, json_mode: bool = False, system: str | None = None, max_tokens: int = 200,
            timeout: int = 150) -> tuple[int, dict | str, dict]:
    """One OpenAI-compatible completion through the gateway (11.3), as the roster member. Returns (status, body, headers)."""
    msgs = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": content}]
    body = {"model": model, "messages": msgs, "max_tokens": max_tokens}
    if json_mode:
        body["response_format"] = {"type": "json_object"}
    r = requests.post(f"{GATEWAY_URL}/v1/chat/completions", json=body, timeout=timeout,
                      headers={"Authorization": f"Bearer {documind_tools._id_token(GATEWAY_URL)}"})
    try:
        return r.status_code, r.json(), dict(r.headers)
    except ValueError:
        return r.status_code, r.text[:400], dict(r.headers)

def service(name: str, region: str | None = None) -> dict:
    """A Cloud Run service as deployed: its env, labels, traffic, image and floor - read with gcloud, the way 10.3 did."""
    r = subprocess.run(["gcloud", "run", "services", "describe", name, "--region", region or REGION, "--project", PROJECT_ID,
                        "--format=json"], capture_output=True, text=True)
    if r.returncode != 0:
        return {}
    j = json.loads(r.stdout)
    c = j["spec"]["template"]["spec"]["containers"][0]
    return {"env": {e["name"]: e.get("value", "") for e in c.get("env", [])}, "image": c.get("image"),
            "labels": (j["metadata"].get("labels") or {}), "traffic": j.get("status", {}).get("traffic", []),
            "url": j.get("status", {}).get("url"), "sa": j["spec"]["template"]["spec"].get("serviceAccountName"),
            "annotations": (j["spec"]["template"]["metadata"].get("annotations") or {}),
            "min_instances": (j["spec"]["template"]["metadata"].get("annotations") or {}).get("autoscaling.knative.dev/minScale", "0")}

# The usage rows the API logs - the ONE shape every observability consumer reads (12.3, tenant_daily). On the lean
# lane they live in Cloud Logging; this reads the last few for a surface, newest first.
def usage_rows(minutes: int = 15, limit: int = 20, event: str = "query", service_name: str = "documind-api") -> list[dict]:
    since = (datetime.datetime.now(datetime.timezone.utc) - datetime.timedelta(minutes=minutes)).strftime("%Y-%m-%dT%H:%M:%SZ")
    r = subprocess.run(["gcloud", "logging", "read",
                        f'resource.type="cloud_run_revision" AND resource.labels.service_name="{service_name}" '
                        f'AND jsonPayload.event="{event}" AND timestamp>="{since}"',
                        "--project", PROJECT_ID, "--limit", str(limit), "--format=json"], capture_output=True, text=True)
    try:
        return [e["jsonPayload"] for e in json.loads(r.stdout or "[]")]
    except ValueError:
        return []

# THE TWENTY LINES THAT MATTER, FROM THE CLONE. Module 12's notebooks are where the kit's files come from (the heredoc
# cells at the end of each notebook are what extract_documind.py reads), so the walls stay there and the story reads
# the file the lane actually runs, around one line, with the file's length beside it.
def excerpt(rel: str, needle: str, before: int = 0, after: int = 14) -> str:
    lines = open(f"{KIT}/deploy/{rel}", encoding="utf-8").read().splitlines()
    i = next(n for n, l in enumerate(lines) if needle in l)
    lo, hi = max(0, i - before), min(len(lines), i + after)
    return f"# {rel}:{lo + 1}-{hi}  ({len(lines)} lines)\n" + "\n".join(lines[lo:hi])

def gcloud(*args: str) -> str:
    """One gcloud read, as the notebook's account, stdout only."""
    r = subprocess.run(["gcloud", *args, "--project", PROJECT_ID], capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else f"(gcloud: {r.stderr.strip()[:200]})"

gcs = storage.Client(project=PROJECT_ID)

sys.path.insert(0, f"{KIT}/deploy/evals")               # run_eval, usage_rows, judge
sys.path.insert(0, f"{KIT}/deploy/services/rag-api")    # the API's own modules, for the excerpts and the pure functions
print("helpers: api(), api_get(), gateway(), service(), usage_rows(), excerpt(), gcloud(); the kit's evals/ and rag-api/ on sys.path")


## Cell 2: The state, read
What Terraform recorded, from the tfstate bucket.


In [ ]:
from collections import Counter

# THE STATE, READ. Terraform's record of the project lives in the tfstate bucket (backend.tf: gs://PROJECT-tfstate under
# the prefix documind/env). Nothing in it was clicked: every resource below was declared in a 12.x heredoc and applied
# by make up. Reading the state is one half of "the code and the project agree"; make drift, in Cloud Shell, is the
# other half - an EMPTY plan after apply, which is the only proof that counts.
blob = gcs.bucket(TFSTATE).blob("documind/env/default.tfstate")
assert blob.exists(), f"no state at gs://{TFSTATE}/documind/env/default.tfstate - has make up run on this project?"
state = json.loads(blob.download_as_text())
resources = [r for r in state.get("resources", []) if r.get("mode") == "managed"]
print(f"terraform {state.get('terraform_version')} | serial {state.get('serial')} | {len(resources)} managed resources")
for t, n in sorted(Counter(r["type"] for r in resources).items(), key=lambda kv: -kv[1])[:16]:
    print(f"  {n:>3}  {t}")
print()
for prefix in ("google_service_account", "google_storage_bucket", "google_secret_manager_secret", "google_billing_budget",
               "google_monitoring_alert_policy", "google_cloud_scheduler_job", "google_cloud_run_v2_job"):
    names = sorted(r["name"] for r in resources if r["type"] == prefix)
    print(f"  {prefix:38} {len(names):>2}  {', '.join(names)[:90]}")
sas = sorted(r["name"] for r in resources if r["type"] == "google_service_account")
print("\naccounts declared (sa.tf, gateway.tf, off.tf):", sas)
assert len(sas) >= 8, "the lane's accounts are missing from the state: this is not the project make up ran on"


## Cell 3: The project, read back
Accounts, buckets, secrets, the budget - from the project, so the state and the project agree.


In [ ]:
# THE PROJECT, READ BACK. The same facts from the other side - IAM, Storage, Secret Manager, Billing - so the state and
# the project agree. Twelve accounts, not the three the first version of this lesson counted; six secrets, and which
# hold a version (make secrets fills cookie-secret; hf-token is yours to add for 10.5 and 11.1; the master key is the
# full profile's); six buckets with their rules, the audit bucket locked for five years; the budget's four thresholds.
accounts = sorted(a.split("@")[0] for a in gcloud("iam", "service-accounts", "list", "--format=value(email)").split() if "documind" in a)
print(f"{len(accounts)} accounts:", ", ".join(accounts))
assert "documind-api-sa" in accounts and "documind-outsider-sa" in accounts, accounts
print()
for line in gcloud("storage", "buckets", "list", f"--filter=name~^{PROJECT_ID}-", "--format=value(name,storageClass,retentionPolicy.retentionPeriod)").splitlines():
    print("  bucket ", line)
audit = gcs.get_bucket(f"{PROJECT_ID}-audit")
print(f"  audit   retention {audit.retention_period} s ({audit.retention_period / 31536000:.0f} years), locked={audit.retention_policy_locked}")
assert audit.retention_policy_locked, "storage.tf locks the audit bucket: nothing may delete a record, Terraform included"
print()
for s in gcloud("secrets", "list", "--format=value(name)").split():
    versions = gcloud("secrets", "versions", "list", s, "--filter=state=ENABLED", "--format=value(name)").split()
    print(f"  secret  {s:22} {len(versions)} enabled version(s)")
billing = gcloud("billing", "projects", "describe", PROJECT_ID, "--format=value(billingAccountName)").split("/")[-1]
raw = gcloud("billing", "budgets", "list", f"--billing-account={billing}", "--format=json")
try:
    for b in json.loads(raw):
        print("  budget ", b.get("displayName"), "units", (b.get("amount", {}).get("specifiedAmount") or {}).get("units"),
              "thresholds", [r.get("thresholdPercent") for r in b.get("thresholdRules", [])])
except ValueError:
    print("  budget ", raw[:160] or "(not visible to this account)")


## Cell 4: Two profiles, one tree


In [ ]:
import glob, re

# TWO PROFILES, ONE TREE. Every full-profile resource is count-gated in the same files (variables.tf: profile = lean |
# full, local.full), so the lean lane and the full deployment are one plan with one variable flipped - never two trees
# that drift. Counted from the clone: which files hold gates, and therefore what lean does not have. make plan
# PROFILE=full is printed and never applied here: it would create Vector Search, Cloud SQL, BigQuery and a cluster,
# and the bill with them (the lane map prices each).
gated = {}
for f in sorted(glob.glob(f"{KIT}/deploy/terraform/*.tf")):
    n = len(re.findall("local.full", open(f, encoding="utf-8").read()))
    if n:
        gated[os.path.basename(f)] = n
print("count-gated files:", gated)
assert "vector.tf" in gated and "cloudsql.tf" in gated and "sink.tf" in gated
print("\nlean lacks: Vector Search (vector.tf), the BigQuery sink and views (sink.tf), Dataplex (dataplex.tf), Cloud SQL (cloudsql.tf),")
print("            Cloud Deploy (clouddeploy.tf), the Gemini quota override (quota.tf), GKE (gke.tf), Spanner, the gateway's database")
mk = open(f"{KIT}/deploy/Makefile", encoding="utf-8").read()
print("\nwhat make up runs, in order (Cloud Shell, never a notebook):")
print(chr(10).join(l for l in mk.split("up: guard-project", 1)[1].split(chr(10) * 2, 1)[0].splitlines()[:5]))
print("\nmake drift PROJECT=<id> ...   # the plan after apply must be EMPTY; a resource still 'to be replaced' is drift, named")
print("make plan  PROJECT=<id> PROFILE=full ...   # the same tree with the variable flipped: printed, not applied")


## Cell 5: The controls


In [ ]:
# WHAT 12.1 PROVISIONS NOW THAT APRIL'S DID NOT: the cost controls (Module 11's evening). The region's L4 quota capped
# at 1 by a consumer override, read from Service Usage; the nightly off job and its schedule; the alert policies, the
# two "left warm" alarms included. Infrastructure is not only what serves - it is what stops the bill.
r = subprocess.run([sys.executable, f"{KIT}/deploy/services/slm/gpu_quota.py", "--project", PROJECT_ID, "--region", REGION],
                   capture_output=True, text=True)
print((r.stdout or r.stderr)[-900:])
print("scheduler:", gcloud("scheduler", "jobs", "describe", "documind-off-nightly", "--location", REGION, "--format=value(state,schedule,timeZone)")
      or "documind-off-nightly not created yet: make plan, then the apply (terraform/off.tf)")
print("job      :", gcloud("run", "jobs", "describe", "documind-off", "--region", REGION, "--format=value(metadata.name,status.executionCount)")
      or "documind-off not created yet")
sess = AuthorizedSession(creds)
pols = sess.get(f"https://monitoring.googleapis.com/v3/projects/{PROJECT_ID}/alertPolicies", timeout=60).json().get("alertPolicies", [])
print("policies :", " | ".join(p["displayName"] for p in pols) or "(none visible)")
print("\nthe lane at rest bills about Rs 200 a month; the three lines above are what keeps it there")


## Where this goes
- **12.2** is the API these accounts and buckets exist for; its `/version` names the image the state does not know about.
- **12.6** owns the switches the controls above are for; **12.7** ships through a candidate revision on this same lane.

## ✅ Lesson 12.1 complete
- ✅ The state read from the tfstate bucket: the managed resources by type, the accounts declared
- ✅ The project read back: accounts, buckets and the audit lock, secrets and their versions, the budget's thresholds
- ✅ The two profiles counted from the clone; what lean lacks, named; `make drift` as the proof
- ✅ The cost controls read: the L4 cap, the nightly job, the alert policies


## The files this lesson owns
Everything below is what `deploy/extract_documind.py` reads: the eleven heredocs that become `deploy/terraform/*.tf`, the API-enable block and the apply block. **The notebook is the source.** To change a deployed file, change it on disk, run `python tools/readopt.py <this notebook> VAR <file>` to push it back into the heredoc, then `python deploy/extract_documind.py --check` - editing `deploy/` alone works until the next extraction quietly reverts you. The cells are unchanged by the rebuild; the story above reads them from the clone.


In [ ]:
PROJECT_ID = 'documind-ai-YOUR-ID'  # CHANGE THIS
REGION = 'us-central1'
INDIA_REGION = 'asia-south1'

# NOT an f-string. As f'''...''' the extractor returns the body raw, so
# {PROJECT_ID} was never substituted AND every \\ stayed doubled - a
# literal backslash instead of a line continuation, which breaks the
# whole enable block. Shell $PROJECT needs no interpolation at all.
ENABLE_APIS = '''
gcloud config set project $PROJECT

# Two calls, not one: the Service Usage API takes at most 20 services per request
# (SU_MAX_BATCH_SIZE_EXCEEDED), and this list is 35 - Vision, Natural Language and
# Translation joined it with Module 9 (9.3). Both are idempotent.
gcloud services enable \\
  run.googleapis.com \\
  compute.googleapis.com \\
  vpcaccess.googleapis.com \\
  pubsub.googleapis.com \\
  artifactregistry.googleapis.com \\
  secretmanager.googleapis.com \\
  firestore.googleapis.com \\
  storage.googleapis.com \\
  aiplatform.googleapis.com \\
  documentai.googleapis.com \\
  vision.googleapis.com \\
  language.googleapis.com \\
  translate.googleapis.com \\
  speech.googleapis.com \\
  texttospeech.googleapis.com \\
  dlp.googleapis.com \\
  iap.googleapis.com \\
  iamcredentials.googleapis.com \\
  cloudbuild.googleapis.com

gcloud services enable \\
  cloudtrace.googleapis.com \\
  monitoring.googleapis.com \\
  logging.googleapis.com \\
  billingbudgets.googleapis.com \\
  bigquery.googleapis.com \\
  discoveryengine.googleapis.com \\
  dataplex.googleapis.com \\
  sqladmin.googleapis.com \\
  eventarc.googleapis.com \\
  workflows.googleapis.com \\
  cloudscheduler.googleapis.com \\
  cloudfunctions.googleapis.com \\
  modelarmor.googleapis.com \\
  cloudbilling.googleapis.com \\
  cloudresourcemanager.googleapis.com \\
  serviceusage.googleapis.com
'''
print(ENABLE_APIS)


In [ ]:
BACKEND_TF = '''
terraform {
  required_version = ">= 1.9.0"
  required_providers {
    google      = { source = "hashicorp/google",      version = "~> 6.15" }
    google-beta = { source = "hashicorp/google-beta", version = "~> 6.15" }
    # cloudsql.tf (12.8): the checkpointer's database password is generated here and lives
    # only in state and in Secret Manager - never in a variable, a notebook or a shell history.
    random      = { source = "hashicorp/random",      version = "~> 3.6" }
  }
  backend "gcs" {
    prefix = "documind/env"
  }
}

provider "google" {
  project = var.project_id
  region  = var.region
}
provider "google-beta" {
  project = var.project_id
  region  = var.region
}
'''
with open('backend.tf', 'w') as f: f.write(BACKEND_TF)
print('backend.tf written')
print()
print('One-time: create the state bucket (NOT managed by tf itself)')
print(f'  gsutil mb -l {REGION} -b on gs://{PROJECT_ID}-tfstate')
print(f'  gsutil versioning set on gs://{PROJECT_ID}-tfstate')


In [ ]:
VARS_TF = '''
variable "project_id"  { type = string }
variable "region"      {
  type    = string
  default = "us-central1"
}
variable "india_region"{
  type    = string
  default = "asia-south1"
}
variable "env"         {
  type    = string
  default = "dev"
}  # dev | staging | prod

# Which of the kit's two shapes this project gets. `lean` is the Module 4 lane and nothing
# else: Cloud Run, Firestore with its vector indexes, one Document AI processor, the uploads
# bucket, the service accounts, secrets, budget and alerts - close to nothing while idle.
# `full` adds what Module 12 teaches on top: Vector Search and its endpoint (the ANN tier,
# the one line item that bills by the hour), the Spanner Graph trial, the Cloud SQL
# checkpointer, GKE, the BigQuery mirror and Dataplex scan, the log sink and Cloud Deploy.
# Same files, `count` on the difference; flip it and apply again, nothing is thrown away.
variable "profile" {
  type    = string
  default = "lean"           # lean | full
  validation {
    condition     = contains(["lean", "full"], var.profile)
    error_message = "profile must be lean or full."
  }
}

locals {
  full = var.profile == "full"
}

# 11.5's Autopilot cluster on the lean lane, for one hour: make gke-up sets it, make gke-down clears it. Off by default
# because an empty cluster still bills its fee (Rs 6,000 a month for nothing).
variable "gke_cluster" {
  type    = bool
  default = false
}

# The one repository allowed to impersonate the deploy identity. Without this the
# WIF provider trusts every repository on GitHub, which is the whole internet.
variable "github_repository" {
  type        = string
  default     = "netsetos/agentic-ai-weekend-gcp"
  description = "owner/repo permitted through Workload Identity Federation"
}

variable "github_repository_id" {
  description = "IMMUTABLE numeric id of the GitHub repo. `gh api repos/OWNER/REPO --jq .id`"
  type        = string
  # No default on purpose. A wrong id fails closed - no workflow can authenticate -
  # whereas a wrong NAME can fail open if somebody else owns that name.
  validation {
    condition     = can(regex("^[0-9]+$", var.github_repository_id))
    error_message = "github_repository_id must be the numeric id, not owner/repo."
  }
}

# Which data-residency story this deployment follows. 12.5 reads it to pick the
# Doc AI processor - Layout Parser is US-only, Enterprise OCR runs in asia-south1 -
# and it is the switch that keeps PII in-country without editing code.
variable "residency" {
  type    = string
  default = "india"          # india | us
  validation {
    condition     = contains(["india", "us"], var.residency)
    error_message = "residency must be india or us."
  }
}

# The document lifecycle (12 September 2026, deploy/INDEXING.md). ONE embedding, declared once: the ingest
# worker stamps the pair on every chunk row and rag-api embeds every query with it. make deploy-services reads
# the two outputs below into both services' environments (EMBEDDING_MODEL, EMBEDDING_VERSION), so a query
# vector from one model against document vectors from another cannot happen by drift - a bump is a plan, an
# apply, a rebuild and a reindex (make reembed, the strategy's next phase).
variable "embedding_model" {
  type    = string
  default = "text-embedding-005"
}

variable "embedding_version" {
  type    = string
  default = "1"
}

# How long a retired chunk row stays before the TTL policy in firestore_indexes.tf removes it: the audit window
# and the undo window. The worker stamps expire_at from it (RETENTION_DAYS, read from the output by
# make deploy-services); nothing else on the lane deletes a chunk.
variable "retention_days" {
  type    = number
  default = 30
  validation {
    condition     = var.retention_days >= 1 && var.retention_days <= 3650
    error_message = "retention_days must be between 1 and 3650."
  }
}

output "embedding_model" {
  description = "EMBEDDING_MODEL for the ingest worker and rag-api"
  value       = var.embedding_model
}

output "embedding_version" {
  description = "EMBEDDING_VERSION for the ingest worker and rag-api"
  value       = var.embedding_version
}

output "retention_days" {
  description = "RETENTION_DAYS for the ingest worker: expire_at = superseded_at + this"
  value       = var.retention_days
}
'''

SA_TF = '''
# Three service accounts, one per service, least-privilege from day zero.
resource "google_service_account" "ui" {
  account_id   = "documind-ui-sa"
  display_name = "DocuMind UI (Streamlit on Cloud Run)"
}
resource "google_service_account" "api" {
  account_id   = "documind-api-sa"
  display_name = "DocuMind API (FastAPI backend)"
}
# The ingest worker (12.5). A fourth identity rather than a reuse of api-sa:
# it reads uploads, writes Firestore, and calls Doc AI and Vertex. It has no
# business reading a secret or invoking another Cloud Run service, and the
# cheapest time to say so is before it exists.
resource "google_service_account" "ingest" {
  account_id   = "documind-ingest-sa"
  display_name = "DocuMind ingest worker"
}

resource "google_service_account" "admin" {
  account_id   = "documind-admin-sa"
  display_name = "DocuMind Admin + Observability"
}

# The chat service (12.8). Its own identity, not a reuse of ui-sa: it invokes rag-api,
# reads the roster, calls Gemini and - since the checkpointer (cloudsql.tf, 8.5) - opens
# one Cloud SQL connection and reads one secret. Nothing else, and the list is here to
# be read. Until 2026-09-05 this service had no account at all and no deploy command.
# It sits on the three golden rosters (deploy/Makefile `roster`) like the UI's account: a
# surface calls the API as itself and forwards the person's assertion when there is one.
# Until 2026-09-09 this account doubled as the eval gate's outsider, and the service's first
# live smoke found its every retrieve refused by the API - correctly. See `outsider` below.
resource "google_service_account" "chat" {
  account_id   = "documind-chat-sa"
  display_name = "DocuMind chat (LangChain agent on Cloud Run)"
}

# The MCP server (7.1-7.2): the lane's agent surface, the way the UI is its human surface.
# Its own identity, like the UI's: it invokes rag-api through the ONE retrieve(), reads the
# roster and the ingest claims, and writes audit rows. Not chat-sa and not ui-sa: two
# services under one identity blur 12.3's audit log.
resource "google_service_account" "mcp" {
  account_id   = "documind-mcp-sa"
  display_name = "DocuMind MCP server (agent surface on Cloud Run)"
}

# The A2A peer (8.4): the agent OUTSIDE the kit. It reaches the lane only through the MCP
# server, as itself, so it needs to invoke documind-mcp and to call Gemini - and nothing else.
# No Firestore, no storage, no rag-api. If this list ever grows, the peer has stopped being
# a peer and become a fifth brain without the chat service's identity rules.
resource "google_service_account" "agent" {
  account_id   = "documind-agent-sa"
  display_name = "DocuMind A2A peer (ADK agent over the MCP server, on Cloud Run)"
}

# The eval gate's OUTSIDER (4.8's run_eval.check_isolation, and the last check of every smoke
# test): an account IAM admits - roles/run.invoker on every service - that sits on no roster,
# so a refusal comes from the roster and not from the network. A fixture, not a service. A
# service account that plays this part cannot also run a service, which is how the chat
# service's first live smoke (2026-09-09) found every retrieve refused, as designed.
resource "google_service_account" "outsider" {
  account_id   = "documind-outsider-sa"
  display_name = "DocuMind eval outsider (IAM admits it, no roster does)"
}

# Signed URLs on Cloud Run need self-impersonation: generate_signed_url signs through the IAM
# signBlob API with the service's own account, and "you need a private key to sign credentials"
# is what the call says without this grant. The UI signs the figure and segment URLs its
# citations open (9.6); the API signs the upload URLs 9.4's Studio hands out - it had no grant
# until Module 9 joined the lane (9 September 2026), so /v1/media/upload-url could never work.
resource "google_service_account_iam_member" "ui_self_impersonate" {
  service_account_id = google_service_account.ui.name
  role               = "roles/iam.serviceAccountTokenCreator"
  member             = "serviceAccount:${google_service_account.ui.email}"
}
resource "google_service_account_iam_member" "api_self_impersonate" {
  service_account_id = google_service_account.api.name
  role               = "roles/iam.serviceAccountTokenCreator"
  member             = "serviceAccount:${google_service_account.api.email}"
}

locals {
  ui_roles = [
    "roles/aiplatform.user",
    "roles/documentai.apiUser",
    "roles/secretmanager.secretAccessor",
    "roles/datastore.user",
    "roles/speech.editor",
    # Without this the frontend cannot call rag-api at all - 12.4's entire chat
    # path is a 403 until it exists. Narrow it to the rag-api service with a
    # google_cloud_run_v2_service_iam_member once the service is Terraform-managed.
    "roles/run.invoker",
    # NO BigQuery roles here, deliberately. The frontend used to carry a second
    # admin dashboard that queried the warehouse directly; it is now a link to the
    # admin service, which runs under admin-sa. Granting ui-sa bigquery.dataViewer
    # would put warehouse credentials in the process that renders user chat, and
    # an XSS there would reach the warehouse - the blast radius this split exists
    # to prevent.
  ]
  api_roles = [
    "roles/aiplatform.user",
    # 12.6: guard.py sanitises prompts and answers against the Model Armor template when ARMOR=on. The role costs
    # nothing while the switch is off, and a revision switched on without it fails closed on every request.
    "roles/modelarmor.user",
    "roles/datastore.user",
    "roles/secretmanager.secretAccessor",
    "roles/logging.logWriter",
    "roles/cloudtrace.agent",
    # retriever.rerank() calls the Discovery Engine semantic ranker
    # (semantic-ranker-fast-004). Without this the rerank step fails and
    # retrieval quietly degrades to unranked vector hits.
    "roles/discoveryengine.viewer",
    # guard.py sanitises every prompt and every answer (12.6). Without this the
    # first sanitize call 403s and the guard fails closed - so the service refuses
    # every question, and the logs blame Model Armor rather than IAM.
    "roles/modelarmor.user",
  ]
  ingest_roles = [
    "roles/storage.objectViewer",     # read the uploaded object, not write it
    "roles/datastore.user",           # documents/ claims + chunks/
    "roles/documentai.apiUser",
    "roles/aiplatform.user",          # embeddings + streaming upserts
    "roles/logging.logWriter",
    "roles/cloudtrace.agent",
    # 12.5's worker scans every chunk before indexing it. Without this the
    # scan 403s, the message is nacked, and every upload lands in the DLQ.
    "roles/dlp.user",
    # ...and writes doc.upload / dlp.finding events to the audit bucket.
    "roles/storage.objectCreator",
  ]
  cicd_roles = [
    "roles/cloudbuild.builds.editor",
    "roles/artifactregistry.writer",
    "roles/clouddeploy.releaser",
    # Cloud Deploy names this SA as the EXECUTION account for RENDER, DEPLOY and
    # VERIFY (clouddeploy.tf). releaser only lets it CREATE a release; without
    # jobRunner the release is cut and then dies at rollout.
    "roles/clouddeploy.jobRunner",
    # ...and the deploy job actuates a Cloud Run service. Google's Cloud Deploy
    # service-account page names exactly this pair for a Cloud Run target:
    # jobRunner plus the runtime's developer role (plus actAs, granted per
    # account below).
    "roles/run.developer",
    # NOT roles/storage.objectAdmin. Cloud Deploy does stage rendered manifests in
    # a GCS bucket, and jobRunner already carries that access. objectAdmin at
    # PROJECT scope would be full object control over every bucket here - which
    # includes the uploads bucket of customer documents and the retention-locked
    # audit bucket. Adding it "because Cloud Deploy touches storage" is exactly
    # the reflex this file exists to argue against.
    #
    # NOT roles/logging.logWriter either: it is not in the documented set, and the
    # execution jobs log through the Cloud Deploy service agent, not through this
    # identity.
    # NOTE: roles/iam.serviceAccountUser is deliberately NOT here. At project
    # scope it would confer actAs on every service account in the project,
    # including ones created after this file. It is granted per-account below.
  ]

  # The identities this pipeline may act as - a MAP, keyed on static strings.
  #
  # It was a list, and that broke `terraform plan` outright: for_each over
  # google_service_account.*.name keys the resource on a value Terraform cannot
  # know until apply, and Terraform refuses ("The keys of the map or all values
  # in a set of strings must be known values"). The keys below are literals; the
  # unknown part rides in each.value, which is allowed.
  chat_roles = [
    "roles/aiplatform.user",   # Gemini through langchain-google-genai (shared/profile.py)
    "roles/datastore.user",    # the tenant roster (shared/tenancy.py)
    "roles/run.invoker",       # rag-api, through the ONE retrieve() - narrow it per service once Terraform-managed
    "roles/logging.logWriter",
    "roles/cloudtrace.agent",
    # secretmanager.secretAccessor and cloudsql.client are granted in cloudsql.tf, on the one
    # secret and for the one instance the checkpointer needs - not project-wide here.
  ]

  mcp_roles = [
    "roles/run.invoker",       # rag-api, through the ONE retrieve() - narrow it per service once Terraform-managed
    "roles/datastore.viewer",  # the roster (tenancy), the ingest claims and chunk counts - reads only
    "roles/logging.logWriter",
    "roles/cloudtrace.agent",
    "roles/storage.objectCreator",   # audit rows, when AUDIT_BUCKET is set (full profile)
  ]

  outsider_roles = [
    "roles/run.invoker",       # may knock on every service; nothing else, and no roster
  ]

  agent_roles = [
    "roles/run.invoker",       # documind-mcp, and only that - narrow it per service once Terraform-managed
    "roles/aiplatform.user",   # Gemini, through ADK
    "roles/logging.logWriter",
    "roles/cloudtrace.agent",
  ]

  cicd_actas = {
    api    = google_service_account.api.name
    ui     = google_service_account.ui.name
    ingest = google_service_account.ingest.name
    admin  = google_service_account.admin.name
    chat   = google_service_account.chat.name
    mcp    = google_service_account.mcp.name
    agent  = google_service_account.agent.name
    # ...and itself. clouddeploy.tf names THIS account as the RENDER/DEPLOY/VERIFY
    # execution account, and Cloud Deploy requires actAs on the execution account
    # even when it is the caller. Without this the release is cut and then dies at
    # rollout, naming the account the operator deliberately chose - which reads as
    # "wrong account" and is really "missing self-actAs".
    cicd   = google_service_account.cicd.name
  }
  admin_roles = [
    "roles/monitoring.viewer",
    "roles/logging.viewer",
    "roles/datastore.user",
    "roles/bigquery.jobUser",
    "roles/run.developer",   # budget-guard fn calls run_v2.update_service to set min_instances=0
    # bigquery.jobUser lets admin-sa RUN a query; it does not let it READ a
    # table. Both are needed, and the failure without dataViewer is a
    # permission error on the first SELECT rather than at deploy time.
    "roles/bigquery.dataViewer",
    # CSV/PDF exports from the admin dashboard land in the audit bucket.
    "roles/storage.objectCreator",
    # admin/dlp.py calls inspect_content and deidentify_content. Without this the
    # audit page raises 403 on the first scan - and the failure looks like a bad
    # request rather than a missing role, because DLP reports it per-item.
    "roles/dlp.user",
  ]
}

# The identity GitHub Actions impersonates through Workload Identity Federation.
# 12.7 builds the pipeline; the account belongs here, with the other four, because
# the interesting part is the SHAPE of its permissions.
#
# Be honest about that shape, because the earlier version of this comment was not.
# It claimed this account "cannot read a secret or a customer document". It could:
# roles/iam.serviceAccountUser was granted at PROJECT scope, which is actAs on
# every service account in the project, so a build submitted as documind-api-sa
# reached Secret Manager and Firestore in one hop.
#
# A deploy identity that deploys a service running as X must be able to act as X -
# that is inherent, not a bug. What is fixable is the BLAST RADIUS: actAs is now
# granted per service account, on the four runtime identities this pipeline
# actually deploys (see cicd_actas), so the list is enumerated, reviewable, and
# does not silently include the next service account somebody adds.
resource "google_service_account" "cicd" {
  account_id   = "sa-documind-cicd"
  display_name = "DocuMind CI/CD (GitHub Actions via WIF)"
}

resource "google_project_iam_member" "cicd" {
  for_each = toset(local.cicd_roles)
  project  = var.project_id
  role     = each.value
  member   = "serviceAccount:${google_service_account.cicd.email}"
}

# actAs, scoped to named accounts - NOT project-wide.
resource "google_service_account_iam_member" "cicd_actas" {
  for_each           = local.cicd_actas
  service_account_id = each.value
  role               = "roles/iam.serviceAccountUser"
  member             = "serviceAccount:${google_service_account.cicd.email}"
}

resource "google_project_iam_member" "ingest" {
  for_each = toset(local.ingest_roles)
  project  = var.project_id
  role     = each.value
  member   = "serviceAccount:${google_service_account.ingest.email}"
}

resource "google_project_iam_member" "mcp" {
  for_each = toset(local.mcp_roles)
  project  = var.project_id
  role     = each.value
  member   = "serviceAccount:${google_service_account.mcp.email}"
}

resource "google_project_iam_member" "outsider" {
  for_each = toset(local.outsider_roles)
  project  = var.project_id
  role     = each.value
  member   = "serviceAccount:${google_service_account.outsider.email}"
}

resource "google_project_iam_member" "agent" {
  for_each = toset(local.agent_roles)
  project  = var.project_id
  role     = each.value
  member   = "serviceAccount:${google_service_account.agent.email}"
}

resource "google_project_iam_member" "ui" {
  for_each = toset(local.ui_roles)
  project  = var.project_id
  role     = each.value
  member   = "serviceAccount:${google_service_account.ui.email}"
}
resource "google_project_iam_member" "api" {
  for_each = toset(local.api_roles)
  project  = var.project_id
  role     = each.value
  member   = "serviceAccount:${google_service_account.api.email}"
}
resource "google_project_iam_member" "chat" {
  for_each = toset(local.chat_roles)
  project  = var.project_id
  role     = each.value
  member   = "serviceAccount:${google_service_account.chat.email}"
}
resource "google_project_iam_member" "admin" {
  for_each = toset(local.admin_roles)
  project  = var.project_id
  role     = each.value
  member   = "serviceAccount:${google_service_account.admin.email}"
}
'''
with open('variables.tf', 'w') as f: f.write(VARS_TF)
with open('sa.tf', 'w') as f: f.write(SA_TF)
print('variables.tf + sa.tf written')


In [ ]:
NETWORK_TF = '''
resource "google_compute_network" "vpc" {
  name                    = "documind-vpc"
  auto_create_subnetworks = false
}
resource "google_compute_subnetwork" "subnet" {
  name          = "documind-subnet"
  network       = google_compute_network.vpc.id
  region        = var.region
  ip_cidr_range = "10.20.0.0/20"
  private_ip_google_access = true
}
resource "google_vpc_access_connector" "conn" {
  name          = "documind-vpc"
  region        = var.region
  network       = google_compute_network.vpc.name
  ip_cidr_range = "10.8.0.0/28"
  min_instances = 2
  max_instances = 6
}
'''

REGISTRY_TF = '''
resource "google_artifact_registry_repository" "docker" {
  repository_id = "documind"
  format        = "DOCKER"
  location      = var.region
  description   = "DocuMind container images"
  cleanup_policies {
    id     = "keep-recent"
    action = "KEEP"
    most_recent_versions { keep_count = 20 }
  }
  cleanup_policies {
    id     = "delete-old"
    action = "DELETE"
    condition { older_than = "2592000s" }  # 30 days
  }
}
'''

FIRESTORE_TF = '''
resource "google_firestore_database" "main" {
  project     = var.project_id
  name        = "(default)"
  location_id = var.india_region   # DPDPA-aligned data residency
  type        = "FIRESTORE_NATIVE"
  concurrency_mode                = "OPTIMISTIC"
  app_engine_integration_mode     = "DISABLED"
  point_in_time_recovery_enablement = "POINT_IN_TIME_RECOVERY_ENABLED"
  delete_protection_state         = "DELETE_PROTECTION_ENABLED"

  # A database that already existed - Module 4's notebooks create (default) in whichever
  # region the learner chose - is ADOPTED by `make adopt-firestore`, region and all. The
  # residency above applies to a database this kit creates; a location change on an
  # adopted one would otherwise plan a destroy of the corpus.
  lifecycle {
    ignore_changes = [location_id, type, concurrency_mode, app_engine_integration_mode]
  }
}
'''
for name, content in [('network.tf', NETWORK_TF), ('registry.tf', REGISTRY_TF), ('firestore.tf', FIRESTORE_TF)]:
    with open(name, 'w') as f: f.write(content)
print('network.tf + registry.tf + firestore.tf written')


In [ ]:
STORAGE_TF = '''
resource "google_storage_bucket" "uploads" {
  name          = "${var.project_id}-uploads"
  location      = var.india_region   # in-region residency (best practice, not a DPDP mandate)
  force_destroy = false
  uniform_bucket_level_access = true
  public_access_prevention    = "enforced"
  versioning { enabled = true }
  lifecycle_rule {
    condition { age = 90 }
    action    {
      type          = "SetStorageClass"
      storage_class = "NEARLINE"
    }
  }
  lifecycle_rule {
    condition { age = 365 }
    action    { type = "Delete" }
  }
  cors {
    origin          = ["https://documind.example.com"]
    method          = ["GET", "PUT", "POST"]
    response_header = ["Content-Type"]
    max_age_seconds = 3600
  }
}

resource "google_storage_bucket" "tts_cache" {
  name          = "${var.project_id}-tts-cache"
  location      = var.india_region
  uniform_bucket_level_access = true
  lifecycle_rule {
    condition { age = 30 }
    action    { type = "Delete" }
  }
}

# 9.4's Media Studio bucket (gap G8). Same shape as tts_cache: india_region, uniform access,
# a 30-day delete rule - generated media is a cache, not a record. The AUDIT row that says it
# existed is the record, and that bucket is retention-locked for five years. The cors block is
# not optional: a browser PUT to a signed URL is a cross-origin request, and without it the
# upload fails in the browser while working perfectly from curl.
resource "google_storage_bucket" "media" {
  name                        = "${var.project_id}-media"
  location                    = var.india_region
  uniform_bucket_level_access = true
  public_access_prevention    = "enforced"
  lifecycle_rule {
    condition { age = 30 }
    action    { type = "Delete" }
  }
  cors {
    origin          = ["https://documind.example.com"]
    method          = ["PUT"]
    response_header = ["Content-Type"]
    max_age_seconds = 3600
  }
}

# Module 10: the tuning dataset is a document (make trainset writes it here, frozen, with its manifest)
# and the tuned Gemma is a file (10.5's GGUF; the Ollama image 11.4 builds copies it from here).
# Versioned and never expiring: a training file that moved is a model nobody can defend. The owner
# writes; the API's account may read (a model_backend that loads from here is Module 11's).
resource "google_storage_bucket" "datasets" {
  name                        = "${var.project_id}-datasets"
  location                    = var.india_region
  force_destroy               = false
  uniform_bucket_level_access = true
  public_access_prevention    = "enforced"
  versioning { enabled = true }
}
resource "google_storage_bucket_iam_member" "api_datasets" {
  bucket = google_storage_bucket.datasets.name
  role   = "roles/storage.objectViewer"
  member = "serviceAccount:${google_service_account.api.email}"
}

resource "google_storage_bucket" "audit" {
  name          = "${var.project_id}-audit"
  location      = var.india_region
  uniform_bucket_level_access = true
  retention_policy {
    retention_period = 157680000   # 5 years -- audit/RBI retention best practice (DPDP Act sets no fixed number)
    is_locked        = true
  }
}

# audit_log.emit refuses to drop an event, so a writer without this grant fails its request
# outright: the worker (doc.upload, dlp.finding) and the API's media router (9.4) write here.
# objectCreator, not objectAdmin - the bucket is retention-locked and nothing may delete.
resource "google_storage_bucket_iam_member" "ingest_audit" {
  bucket = google_storage_bucket.audit.name
  role   = "roles/storage.objectCreator"
  member = "serviceAccount:${google_service_account.ingest.email}"
}
resource "google_storage_bucket_iam_member" "api_audit" {
  bucket = google_storage_bucket.audit.name
  role   = "roles/storage.objectCreator"
  member = "serviceAccount:${google_service_account.api.email}"
}

# UI can read/write uploads + TTS cache only
resource "google_storage_bucket_iam_member" "ui_uploads" {
  bucket = google_storage_bucket.uploads.name
  role   = "roles/storage.objectAdmin"
  member = "serviceAccount:${google_service_account.ui.email}"
}
resource "google_storage_bucket_iam_member" "ui_tts" {
  bucket = google_storage_bucket.tts_cache.name
  role   = "roles/storage.objectAdmin"
  member = "serviceAccount:${google_service_account.ui.email}"
}

# The API signs 9.4's upload URLs into the UPLOADS bucket - the one with the object.finalized
# notification - under the tenant's prefix. A V4 signed URL is authorised AS its signer, so
# the signer needs objectCreator here and nothing more (it never reads uploads; the worker does).
resource "google_storage_bucket_iam_member" "api_uploads" {
  bucket = google_storage_bucket.uploads.name
  role   = "roles/storage.objectCreator"
  member = "serviceAccount:${google_service_account.api.email}"
}

# Media (9.4 / 9.6, gaps G7-G8): rag-api WRITES generated assets and signs upload URLs;
# the UI only READS, through the V4 signed URLs it mints for figure and segment citations -
# a signed URL is authorised as its signer, so the signer needs objectViewer and nothing more.
resource "google_storage_bucket_iam_member" "api_media" {
  bucket = google_storage_bucket.media.name
  role   = "roles/storage.objectAdmin"
  member = "serviceAccount:${google_service_account.api.email}"
}
resource "google_storage_bucket_iam_member" "ui_media" {
  bucket = google_storage_bucket.media.name
  role   = "roles/storage.objectViewer"
  member = "serviceAccount:${google_service_account.ui.email}"
}
'''
with open('storage.tf', 'w') as f: f.write(STORAGE_TF)
print('storage.tf written')


In [ ]:
SECRETS_TF = '''
variable "secret_names" {
  type    = set(string)
  default = [
    "litellm-master-key",
    "cookie-secret",
    "oauth-client-id",
    "oauth-client-secret",
    "openai-fallback-key",
    # 10.5 pushes the tuned Gemma to a private Hub repo with this token, read from Secret Manager in the
    # notebook - never typed into a cell. Created empty; the owner adds a version when there is a token.
    "hf-token",
  ]
}

resource "google_secret_manager_secret" "s" {
  for_each  = var.secret_names
  secret_id = each.value
  replication {
    auto {}
  }
}

resource "google_secret_manager_secret_iam_member" "ui_access" {
  for_each  = google_secret_manager_secret.s
  secret_id = each.value.id
  role      = "roles/secretmanager.secretAccessor"
  member    = "serviceAccount:${google_service_account.ui.email}"
}
resource "google_secret_manager_secret_iam_member" "api_access" {
  for_each  = google_secret_manager_secret.s
  secret_id = each.value.id
  role      = "roles/secretmanager.secretAccessor"
  member    = "serviceAccount:${google_service_account.api.email}"
}
'''
with open('secrets.tf', 'w') as f: f.write(SECRETS_TF)
print('secrets.tf written')
print()
print('After apply: add the ACTUAL secret value with')
print('  echo -n "$(openssl rand -hex 32)" | gcloud secrets versions add cookie-secret --data-file=-')
print('  gcloud secrets versions add oauth-client-id --data-file=client-id.txt')


In [ ]:
BUDGET_TF = '''
data "google_billing_account" "acct" {
  billing_account = var.billing_account_id
}

resource "google_billing_budget" "documind" {
  billing_account = data.google_billing_account.acct.id
  display_name    = "DocuMind monthly budget"

  budget_filter {
    # The Budgets API names projects by NUMBER. The id form was accepted by the plan and
    # refused by the apply ("Precondition check failed"), the first time it ran for real.
    projects = ["projects/${data.google_project.current.number}"]
  }

  amount {
    # In the billing account's own currency: the API refuses any other (an Indian account
    # is INR, and "USD" was refused on the first live apply as an invalid argument). Leave
    # budget_currency empty and the account's currency is used; the amount is then in it.
    specified_amount {
      currency_code = var.budget_currency != "" ? var.budget_currency : null
      units         = var.budget_amount
    }
  }

  threshold_rules { threshold_percent = 0.5 }
  threshold_rules { threshold_percent = 0.8 }
  threshold_rules { threshold_percent = 1.0 }
  threshold_rules {
    threshold_percent = 1.2
    spend_basis       = "FORECASTED_SPEND"
  }

  # Only when there is something to say. A rule with no channels and no topic is the API's own
  # default (emails to the billing admins), and the API stores nothing for it - so it read back
  # as empty and every plan after the first apply wanted to "update" the budget in place, for
  # ever (F33, the Module 10 plan on 10 September). An omitted block is the same behaviour with
  # nothing to drift.
  dynamic "all_updates_rule" {
    for_each = (length(var.alert_channels) > 0 || var.budget_pubsub) ? [1] : []
    content {
      monitoring_notification_channels = var.alert_channels
      disable_default_iam_recipients   = false
      # The Pub/Sub leg is opt-in (budget_pubsub). Under the domain-restricted-sharing org
      # policy (iam.allowedPolicyMemberDomains) the publisher grant below is refused, because
      # the billing budget agent is a Google system account outside the organisation - which
      # is how the first live apply on an organisation project ended. The emails to billing
      # admins (disable_default_iam_recipients = false) need no grant and always go out.
      pubsub_topic                     = var.budget_pubsub ? google_pubsub_topic.budget_alerts[0].id : null
    }
  }
  depends_on = [google_pubsub_topic_iam_member.budget_publisher]
}

resource "google_pubsub_topic" "budget_alerts" {
  count = var.budget_pubsub ? 1 : 0
  name  = "documind-budget-alerts"
}

# Budget notifications are published by Google's billing budget agent, which needs to be
# allowed to publish to the topic; without this the budget is created and never speaks.
resource "google_pubsub_topic_iam_member" "budget_publisher" {
  count  = var.budget_pubsub ? 1 : 0
  topic  = google_pubsub_topic.budget_alerts[0].id
  role   = "roles/pubsub.publisher"
  member = "serviceAccount:billing-budget-alert@system.gserviceaccount.com"
}

variable "billing_account_id" { type = string }
variable "alert_channels"     {
  type    = list(string)
  default = []
}
variable "budget_pubsub" {
  type        = bool
  default     = false
  description = "Also publish budget notifications to a Pub/Sub topic. Needs an IAM grant to a Google system account, which an organisation with domain-restricted sharing refuses."
}
variable "budget_amount" {
  type        = string
  default     = "500"
  description = "Monthly budget, whole units of budget_currency (or of the billing account's currency when that is empty). The lesson's figure was 500 USD; `make up` passes BUDGET_AMOUNT."
}
variable "budget_currency" {
  type        = string
  default     = ""
  description = "ISO 4217 code, or empty for the billing account's currency - the only one the Budgets API accepts."
}
'''
with open('budget.tf', 'w') as f: f.write(BUDGET_TF)
print('budget.tf written')
print()
print('Thresholds hit at 50/80/100% actual + 120% forecasted.')
print('Pub/Sub topic lets you wire a Cloud Function that scales Cloud Run min_instances to 0 on breach.')


In [ ]:
APPLY = '''
# Dry-run first
terraform init -reconfigure -backend-config="bucket=documind-ai-YOUR-ID-tfstate"
terraform plan -out=tfplan \\
  -var=project_id=documind-ai-YOUR-ID \\
  -var=billing_account_id=YOUR-BILLING-ID \\
  -var=github_repository_id=1358872052 \\
  -var=profile=lean

# Apply when plan is green
terraform apply tfplan

# Smoke tests
gcloud iam service-accounts list --filter="email~documind-.*-sa"
gcloud artifacts repositories describe documind --location=us-central1
gcloud firestore databases list
gsutil ls -b gs://documind-ai-YOUR-ID-uploads
gcloud secrets list --filter=name~litellm
gcloud billing budgets list --billing-account=YOUR-BILLING-ID
'''
print(APPLY)
print()
INHERITS = {
    '12.2 RAG API': 'documind-api-sa + VPC connector + Firestore + Vector Search index',
    '12.3 Admin Dashboard': 'documind-admin-sa + audit bucket + budget pubsub topic',
    '12.4 Streamlit UI': 'documind-ui-sa (self-impersonation set), uploads + TTS cache buckets, all 5 secrets',
    'Operations': 'Budget alerts route to Pub/Sub; wire Cloud Function to degrade service on 100% breach',
}
print('EVERYTHING DOWNSTREAM INHERITS:')
for lesson, what in INHERITS.items():
    print(f'  {lesson:30} <- {what}')
